# Module 09 — Practical GPU Benchmarking Laboratory

This is the first complete engineering lab. It combines the GPU concepts from Modules 01–08 into a reproducible T4 performance baseline.

## Lab protocol

Run on one NVIDIA T4. Warm up kernels, use CUDA events for device work, repeat measurements, and record shape, dtype, bytes, FLOPs, latency, memory, and software versions.

In [ ]:
from pathlib import Path
import sys
module_dir = Path.cwd() / 'code' / 'module-09'
if not (module_dir / 'benchmark_lab.py').exists():
    module_dir = Path.cwd()
sys.path.insert(0, str(module_dir))

import torch
from benchmark_lab import (
    allocation_experiment, benchmark_gpu, benchmark_wall, device_report,
    gemm_experiment, launch_experiment, memory_report, projection_experiment,
    require_cuda, transfer_experiment, vector_add_experiment,
)

device = require_cuda('cuda:0')
print('Device:', device)
device_report(device)

## 1. GPU inspection and allocator state

In [ ]:
allocation = allocation_experiment(device)
allocation

## 2. Memory bandwidth

FP32 vector addition has one FLOP and 12 useful bytes per element, so its arithmetic intensity is approximately 0.083 FLOPs/byte.

In [ ]:
vector = vector_add_experiment(device, vector_mib=128, warmup=10, repeats=30)
vector

## 3. GEMM throughput and 4. matrix-shape efficiency

In [ ]:
gemm = gemm_experiment(device, [256, 512, 1024, 2048, 4096], warmup=10, repeats=30)
projections = projection_experiment(device, [1, 4, 16, 32, 64, 128, 256, 512], 4096, warmup=10, repeats=30)
gemm[-1], projections[0], projections[-1]

## 5. Kernel-launch overhead

In [ ]:
launch = launch_experiment(device, elements=1024, count=100, warmup=10, repeats=30)
launch

## 6. CPU–GPU transfers

In [ ]:
transfers = transfer_experiment(device, transfer_mib=128, warmup=5, repeats=20)
transfers

## 7. Build the engineering interpretation

Complete a report with the GPU, CUDA/PyTorch versions, shapes, dtypes, latency, effective bandwidth, achieved TFLOPS, allocator state, and a diagnosis of which measurements are memory-bound, compute-oriented, or launch-sensitive.

The goal is not one impressive number. The goal is a baseline you can compare against when Module 10 introduces a real language-model request.